# SqlPartitionedLoader — partitioned SQL loading

Demonstrates the `SqlPartitionedLoader` for lazy, chunked SQL → Dask DataFrame loading.

| # | Topic |
|---|---|
| 1 | SqlPartitionedLoadRequest — request model |
| 2 | SqlPartitionPlan — inspect the plan before loading |
| 3 | SqlPartitionSpec — individual partition SQL |
| 4 | Offset vs range partitioning strategies |
| 5 | load() with diagnostics |

In [4]:
import sys
import tempfile
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "boti").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

import pandas as pd
from sqlalchemy import Integer, String, create_engine, select
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column

from boti_data import (
    SqlPartitionedLoader,
    SqlPartitionedLoadRequest,
    SqlPartitionPlan,
    SqlPartitionSpec,
)
from boti_data.db.sql_manager import SqlDatabaseConfig

## Setup: create a seed database

We need a database with enough rows to demonstrate partitioning.

In [5]:
class Base(DeclarativeBase):
    pass

class Event(Base):
    __tablename__ = "events"
    id: Mapped[int] = mapped_column(Integer, primary_key=True)
    name: Mapped[str] = mapped_column(String(50))
    value: Mapped[float] = mapped_column(nullable=True)

db_file = Path(tempfile.mktemp(suffix=".db"))
engine = create_engine(f"sqlite:///{db_file}")
Base.metadata.create_all(engine)

with Session(engine) as session:
    for i in range(1000):
        session.add(Event(id=i+1, name=f"event_{i}", value=float(i * 10)))
    session.commit()

config = SqlDatabaseConfig(connection_url=f"sqlite:///{db_file}", query_only=False)
print(f"Database seeded at {db_file} with 1000 rows")


Database seeded at /var/folders/j1/c0fy94996q51rf0nkcvg577m0000gn/T/tmpjidvkf_g.db with 1000 rows


## 1. SqlPartitionedLoadRequest — request model

The request specifies the SQL statement, model, partitioning parameters, and chunk size.

In [6]:
stmt = select(Event)

request = SqlPartitionedLoadRequest(
    statement=stmt,
    model=Event,
    partitioned=True,
    partition_strategy="offset",
    chunk_size=200,
    order_column="id",
    diagnostics=True,
)

print(f"Strategy:       {request.partition_strategy}")
print(f"Chunk size:     {request.chunk_size}")
print(f"Max concurrent: {request.max_concurrent_fetches}")
print(f"Diagnostics:    {request.diagnostics}")


ValidationError: 1 validation error for SqlPartitionedLoadRequest
  Value error, partitioned SQL statements must not include ORDER BY; use order_column instead. [type=value_error, input_value={'statement': <sqlalchemy...00, 'diagnostics': True}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error

## 2. SqlPartitionPlan — inspect the plan

`plan()` computes partition boundaries without fetching data.

In [ ]:
with SqlPartitionedLoader(config) as loader:
    plan = loader.plan(request)
    
    print(f"Plan strategy:    {plan.strategy}")
    print(f"Total rows:       {plan.total_rows}")
    print(f"Num partitions:   {len(plan.partitions)}")
    print(f"Meta dtypes:      {plan.meta_dtypes}")

## 3. SqlPartitionSpec — inspect individual partition SQL

Each partition is a `(sql, params)` tuple that can be executed independently.

In [ ]:
with SqlPartitionedLoader(config) as loader:
    plan = loader.plan(request)
    
    for idx, spec in enumerate(plan.partitions):
        print(f"Partition {idx}: sql={spec.sql[:80]}... params={spec.params}")
        if idx >= 2:
            print(f"  ... ({len(plan.partitions) - 3} more partitions)")
            break

## 4. Offset vs range partitioning strategies

- **Offset partitioning** — uses `LIMIT/OFFSET`; works with any table, supports limits.
- **Range partitioning** — uses `WHERE key >= :lo AND key < :hi`; requires a partition column, more efficient for large offsets.

In [ ]:
# Range partitioning on the id column
range_request = SqlPartitionedLoadRequest(
    statement=stmt,
    model=Event,
    partitioned=True,
    partition_strategy="range",
    partition_column="id",
    chunk_size=300,
)

with SqlPartitionedLoader(config) as loader:
    range_plan = loader.plan(range_request)
    
print(f"Range plan: {len(range_plan.partitions)} partitions, {range_plan.total_rows} rows")
for idx, spec in enumerate(range_plan.partitions):
    print(f"  Partition {idx}: {spec.sql}")


## 5. `load()` with diagnostics

Executes the partition plan and returns a Dask DataFrame (or pandas with `as_pandas=True`).

In [ ]:
with SqlPartitionedLoader(config) as loader:
    result = loader.load(request)

print(f"Result type: {type(result).__name__}")
if hasattr(result, "npartitions"):
    print(f"Partitions:  {result.npartitions}")
print(result.compute().head(10))

In [ ]:
# Cleanup
db_file.unlink(missing_ok=True)

### Summary

- **`SqlPartitionedLoadRequest`** — validated request model for partitioned loading.
- **`SqlPartitionPlan`** — partition boundaries computed without data fetch.
- **`SqlPartitionSpec`** — individual (sql, params) for each partition.
- **Offset vs Range** — offset works universally; range requires a partition column but is more efficient.
- **`loader.load()`** — returns a lazy Dask DataFrame from partitioned SQL fetches.
- Set `diagnostics=True` to enable detailed logging of plan and execution metrics.